# 05 发育阶段比较

比较 E7.5、E7.75 和 E8.0。真实文件下载后逐样本读取；无真实文件时仅验证代码流程。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
from create_sample_data import synthetic
from h5ad_utils import choose_h5ad
synthetic_path = ROOT / 'data' / 'processed' / 'synthetic_test.h5ad'
if not list((ROOT / 'data' / 'external' / 'GSE278603' / 'h5ad').glob('*.h5ad')):
    synthetic(synthetic_path)
H5AD = choose_h5ad(ROOT)
IS_SYNTHETIC = H5AD.name == 'synthetic_test.h5ad'
print('数据文件：', H5AD)
print('注意：' if IS_SYNTHETIC else '状态：', '当前使用完全合成测试数据' if IS_SYNTHETIC else '当前使用真实 GEO 数据')


In [ ]:
import anndata as ad
import pandas as pd
files = sorted((ROOT / 'data' / 'external' / 'GSE278603' / 'h5ad').glob('*.h5ad')) or [H5AD]
rows = []
for path in files:
    a = ad.read_h5ad(path, backed='r')
    if 'stage' in a.obs:
        for stage, count in a.obs['stage'].astype(str).value_counts().items():
            rows.append({'file':path.name,'stage':stage,'n_obs':int(count),'n_vars':a.n_vars})
    else:
        stage = next((s for s in ['E7.75','E7.5','E8.0'] if s in path.name), '字段待确认')
        rows.append({'file':path.name,'stage':stage,'n_obs':a.n_obs,'n_vars':a.n_vars})
    a.file.close()
comparison = pd.DataFrame(rows)
comparison


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei','SimHei','DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
summary = comparison.groupby('stage', sort=False)['n_obs'].sum()
ax = summary.plot.bar(figsize=(7,4), color='#4C78A8')
ax.set_ylabel('观测数量'); ax.set_title(('合成测试：' if IS_SYNTHETIC else '') + '各发育阶段观测数')
plt.tight_layout()
out = ROOT / 'results' / 'figures' / 'notebook_stage_comparison.png'
plt.savefig(out, dpi=180); plt.show()
comparison.to_csv(ROOT / 'results' / 'tables' / 'stage_comparison.csv', index=False)
out
